This notebook was motivated in part by Chollet's "Deep Learning with Python"

# Deep Learning for Handwritten Digit Classification
## A dense neural network, then a convolutional neural network

In the previous notebook we learned the PyTorch workflow on a regression problem: tensors, `nn.Sequential`, `DataLoader`, a loss function, an optimizer, and a training loop. Now we use the same workflow to **classify images**.

The task is the classic **MNIST** dataset: 70,000 grayscale images of handwritten digits, each labeled 0 to 9. In this notebook we will:

1. Load the images with **torchvision**, PyTorch's companion library for computer vision
2. Build, train, and evaluate a **dense** (fully connected) neural network
3. Build, train, and evaluate a **convolutional** neural network (CNN), a type of network designed for images
4. Compare the two

Both models are trained with exactly the same code. Only the model definition and the shape of the input change.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision

torch.set_printoptions(sci_mode=False)   # print small numbers as 0.0001, not 1.0000e-04

print('PyTorch version:', torch.__version__)
print('torchvision version:', torchvision.__version__)

As before, we pick the best available device. Everything in this notebook also runs fine on a CPU.

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print('Using device:', device)

# The MNIST dataset

70,000 grayscale images of handwritten digits, each 28×28 pixels, labeled 0 to 9: 60,000 for training and 10,000 for testing.

Load the training and test sets. The first time, this downloads the data into a folder called `data`.

In [ ]:
train_set = torchvision.datasets.MNIST(root='data', train=True, download=True)
test_set = torchvision.datasets.MNIST(root='data', train=False, download=True)

The images and labels are stored as tensors in the `.data` and `.targets` attributes.

In [ ]:
train_images = train_set.data
train_labels = train_set.targets
test_images = test_set.data
test_labels = test_set.targets

In [ ]:
train_images.shape

In [ ]:
len(train_labels)

In [ ]:
train_labels

In [ ]:
test_images.shape

In [ ]:
len(test_labels)

In [ ]:
test_labels

Each image is a 28×28 grid of integers from 0 (white) to 255 (black):

In [ ]:
train_images.dtype, train_images.min(), train_images.max()

In [ ]:
digit = train_images[4]
digit_label = train_labels[4]

plt.imshow(digit, cmap=plt.cm.binary)
plt.show()
print(digit_label)

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_images[i], cmap=plt.cm.binary)
    ax.set_title(train_labels[i].item())
    ax.axis('off')
fig.tight_layout()
plt.show()

# Prepare the data

Two steps:

1. **Scale** the pixel values from integers 0–255 to floats between 0 and 1. Neural networks train much better on small floating-point inputs.
2. **Reshape** each image to the shape the model expects. A dense layer takes a flat vector, so each 28×28 image becomes a row of 784 numbers.

(Later, for the CNN, we'll reshape the images differently.)

Note that we don't standardize the pixels the way we standardized the housing features. Pixels are already all on the same scale, so dividing by 255 is enough.

In [ ]:
train_images_flat = train_images.reshape(60000, 28 * 28).float() / 255
test_images_flat = test_images.reshape(10000, 28 * 28).float() / 255

train_images_flat.shape, train_images_flat.dtype

In [ ]:
train_images_flat.min(), train_images_flat.max()

### The labels stay as integers

The labels are the integers 0 to 9, and we leave them that way. The loss function we'll use for classification takes the integer class labels directly.

In [ ]:
train_labels[4], train_labels.dtype

### Batches with `DataLoader`

Just as in the previous notebook, `TensorDataset` pairs each image with its label, and `DataLoader` serves up shuffled mini-batches. This time we use batches of 128 images.

For 60,000 training images and a batch size of 128, one epoch is 60000 / 128 ≈ 469 batches.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

train_loader = DataLoader(TensorDataset(train_images_flat, train_labels), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(test_images_flat, test_labels), batch_size=1000)

len(train_loader)

In [ ]:
images_batch, labels_batch = next(iter(train_loader))
images_batch.shape, labels_batch.shape

# Part 1: A dense neural network

## Define the model

The same kind of `nn.Sequential` network as for housing prices, with two changes:

* The input has 784 features (one per pixel), and there's one hidden layer of 512 units.
* The final layer has **10 outputs**, one score per digit, instead of a single number.

### From scores to probabilities: softmax

In the first notebook, for two classes, the model output one number, and the **sigmoid** function turned it into a probability. With 10 classes, the **softmax** function turns the 10 scores $z_0, \dots, z_9$ into 10 probabilities that are positive and add up to 1:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

The raw scores, before softmax, are called **logits**. The model outputs logits, with **no softmax layer at the end**: the loss function below applies softmax itself, in a way that's more numerically stable. When we want probabilities, we apply softmax ourselves.

In [ ]:
torch.manual_seed(42)

dense_network = nn.Sequential(
    nn.Linear(28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)

dense_network = dense_network.to(device)
dense_network

How many parameters? The first layer has 784 weights for each of its 512 units, plus one bias per unit:

In [ ]:
(28 * 28 + 1) * 512

In [ ]:
(512 + 1) * 10

It's handy to see each layer's output shape and number of parameters. This small helper function sends a sample batch through the model one layer at a time and prints each layer's output shape and number of parameters:

In [ ]:
def summary(model, sample_input):
    x = sample_input.to(device)
    print(f"{'Layer':12s} {'Output shape':>20s} {'Params':>10s}")
    print('-' * 44)
    for layer in model:
        x = layer(x)
        n_params = sum(p.numel() for p in layer.parameters())
        print(f"{type(layer).__name__:12s} {str(tuple(x.shape)):>20s} {n_params:>10,d}")
    print('-' * 44)
    print(f"Total params: {sum(p.numel() for p in model.parameters()):,d}")

summary(dense_network, images_batch[:1])

## Choose the loss function and optimizer

* **Cross-entropy** is the standard loss for classification. For each image it's $-\log p_{\text{correct digit}}$: small when the model gives the correct digit a high probability, and large when it doesn't. (It's the same idea as the binary cross-entropy from the first notebook, extended to 10 classes.) `nn.CrossEntropyLoss` takes the model's logits and the integer labels.
* **Adam**, the optimizer that worked well in the previous notebook.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(dense_network.parameters(), lr=0.001)

## Train the model

This is the same training loop as in the previous notebook. For every batch:
1. **forward pass**: run the images through the model to get the logits
2. **loss**: compare them with the true labels
3. `optimizer.zero_grad()`: clear the old gradients
4. `loss.backward()`: compute the gradients
5. `optimizer.step()`: update the parameters

The one addition is that we also count how many images were classified correctly, to report the **accuracy**. The predicted digit is the one with the highest score: `outputs.argmax(dim=1)`. (We don't need softmax for this: the highest score always has the highest probability.)

In [ ]:
def train(model, loader, loss_fn, optimizer, epochs):
    history = {'loss': [], 'accuracy': []}

    for epoch in range(epochs):
        model.train()
        total_loss, total_correct, total_seen = 0.0, 0, 0

        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)            # 1. forward pass
            loss = loss_fn(outputs, labels)    # 2. loss

            optimizer.zero_grad()              # 3. clear old gradients
            loss.backward()                    # 4. compute gradients
            optimizer.step()                   # 5. update parameters

            total_loss += loss.item() * len(labels)
            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
            total_seen += len(labels)

        history['loss'].append(total_loss / total_seen)
        history['accuracy'].append(total_correct / total_seen)
        print(f"Epoch {epoch + 1}/{epochs}   loss: {history['loss'][-1]:.4f}   accuracy: {history['accuracy'][-1]:.4f}")

    return history

Two settings control training:

1. `epochs`: the number of times the algorithm goes through the entire training set
2. the batch size (set in the `DataLoader` above): the number of training samples the model uses before each update of the weights

In [ ]:
dense_history = train(dense_network, train_loader, loss_fn, optimizer, epochs=5)

In [ ]:
epochs = range(1, len(dense_history['loss']) + 1)

plt.plot(epochs, dense_history['loss'], 'bo', label='Training loss')
plt.plot(epochs, dense_history['accuracy'], 'b', label='Training accuracy')
plt.title('Training loss vs accuracy')
plt.xlabel('Epochs')
plt.ylabel('Loss/Accuracy')
plt.xticks(epochs)
plt.legend()
plt.show()

## Evaluate the model using the test data

As in the previous notebook, evaluation uses `model.eval()` and `torch.no_grad()`, since we're not training. This version also computes the accuracy.

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += loss_fn(outputs, labels).item() * len(labels)
            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
            total_seen += len(labels)
    return total_loss / total_seen, total_correct / total_seen

In [ ]:
test_loss, test_acc = evaluate(dense_network, test_loader, loss_fn)
print('test_loss:', test_loss)
print('test_acc:', test_acc)

## Making predictions

Let's look at one test image:

In [ ]:
digit = test_images_flat[4]
digit_label = test_labels[4]

plt.imshow(digit.reshape(28, 28), cmap=plt.cm.binary)
plt.show()
print(digit_label)

In [ ]:
digit.shape

The model expects a **batch** of images, so a single image needs shape `(1, 784)`, a batch of one. `unsqueeze(0)` adds that extra dimension at the front.

In [ ]:
digit.unsqueeze(0).shape

In [ ]:
dense_network.eval()
with torch.no_grad():
    logits = dense_network(digit.unsqueeze(0).to(device))

logits

These are the raw scores. To turn them into probabilities (positive, summing to 1), apply softmax:

In [ ]:
probabilities = torch.softmax(logits, dim=1)
probabilities.round(decimals=4)

In [ ]:
probabilities.argmax(dim=1)

The predicted digit is the one with the highest probability (equivalently, the highest score).

Now the first five test images at once:

In [ ]:
with torch.no_grad():
    logits = dense_network(test_images_flat[:5].to(device))

torch.softmax(logits, dim=1).round(decimals=3)

In [ ]:
for p in torch.softmax(logits, dim=1):
    print(p.argmax().item())

In [ ]:
test_labels[:5]

### Predictions for the whole test set

A function that collects the predicted digit for every image in a `DataLoader`:

In [ ]:
def predict(model, loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            predictions.append(outputs.argmax(dim=1).cpu())
    return torch.cat(predictions)

pred_labels = predict(dense_network, test_loader)
len(pred_labels)

In [ ]:
pred_labels[:20], test_labels[:20]

## Confusion matrix and other metrics

scikit-learn's metrics take the true labels and the predicted labels as integer arrays. (`.numpy()` converts a tensor to a NumPy array.)

In the **confusion matrix**, row $i$, column $j$ counts the test images of digit $i$ that the model predicted as digit $j$. The diagonal holds the correct predictions; everything off the diagonal is a mistake.

In [ ]:
import sklearn.metrics

y_true = test_labels.numpy()
y_pred = pred_labels.numpy()

cm = sklearn.metrics.confusion_matrix(y_true, y_pred)
cm

In [ ]:
print(f"Accuracy: {sklearn.metrics.accuracy_score(y_true, y_pred):.2%}")
print(f"Precision: {sklearn.metrics.precision_score(y_true, y_pred, average='macro'):.2%}")
print(f"Recall: {sklearn.metrics.recall_score(y_true, y_pred, average='macro'):.2%}")

* **Precision** for a digit: of the images the model labeled as that digit, what fraction really were?
* **Recall** for a digit: of the images that really were that digit, what fraction did the model find?

`average='macro'` computes these for each digit separately, then averages the 10 values. The classification report shows them digit by digit:

In [ ]:
print(sklearn.metrics.classification_report(y_true, y_pred))

In [ ]:
sklearn.metrics.ConfusionMatrixDisplay(cm).plot(cmap='Blues', colorbar=False)
plt.title('Dense network')
plt.show()

### Some of the mistakes

In [ ]:
def show_mistakes(y_true, y_pred, title):
    wrong = np.where(y_pred != y_true)[0][:20]
    fig, axes = plt.subplots(2, 10, figsize=(12, 3.2))
    for ax, i in zip(axes.flat, wrong):
        ax.imshow(test_images[i], cmap=plt.cm.binary)
        ax.set_title(f"pred {y_pred[i]}\ntrue {y_true[i]}", fontsize=9)
        ax.axis('off')
    fig.suptitle(f"{title}: {(y_pred != y_true).sum()} mistakes out of {len(y_true)}")
    fig.tight_layout()
    plt.show()

show_mistakes(y_true, y_pred, 'Dense network')

# Part 2: A convolutional neural network (CNN)

The dense network's first step is to flatten each image into a list of 784 numbers. That throws away the fact that pixels have neighbors: the network has no idea that pixel 30 sits right below pixel 2.

A **convolutional layer** keeps the image as a 2-D grid. It slides small **filters** (for example 3×3) across the image, and at each position computes a weighted sum of the pixels under the filter. Each filter produces a **feature map** showing where in the image its pattern (an edge, a corner, a curve) appears. The same filter weights are used at every position, so:

* a pattern can be detected anywhere in the image
* the layer needs very few parameters

A **max pooling** layer then shrinks each feature map by keeping only the largest value in each 2×2 window, halving the height and width.

## Prepare the data: keep the images as 2-D grids

Convolutional layers take input of shape `(batch, channels, height, width)`. Our images are grayscale, so there's **1 channel**. (A color image would have 3: red, green, blue.)

PyTorch always puts the channel dimension **before** the height and width.

In [ ]:
train_images_cnn = train_images.reshape(60000, 1, 28, 28).float() / 255
test_images_cnn = test_images.reshape(10000, 1, 28, 28).float() / 255

train_images_cnn.shape

In [ ]:
torch.manual_seed(42)

train_loader_cnn = DataLoader(TensorDataset(train_images_cnn, train_labels), batch_size=128, shuffle=True)
test_loader_cnn = DataLoader(TensorDataset(test_images_cnn, test_labels), batch_size=1000)

images_batch, labels_batch = next(iter(train_loader_cnn))
images_batch.shape

## Define the model

* `nn.Conv2d(1, 32, kernel_size=3, padding=1)`: 1 input channel, 32 filters, each 3×3. Without padding, a 3×3 filter can't be centered on the edge pixels, so the output would shrink to 26×26. `padding=1` adds a 1-pixel border of zeros around the image so the output stays 28×28.
* `nn.MaxPool2d(2)`: 2×2 max pooling, so 28×28 becomes 14×14
* `nn.Flatten()`: flattens the 32 feature maps of size 14×14 into one vector of 32 × 14 × 14 = 6,272 numbers
* then dense layers, as before

Every `Linear` layer needs to be told its input size, so we have to work out the 6,272 ourselves. The `summary` function is a good way to check.

In [ ]:
torch.manual_seed(42)

cnn_network = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),

    nn.Flatten(),

    nn.Linear(32 * 14 * 14, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

cnn_network = cnn_network.to(device)
cnn_network

In [ ]:
summary(cnn_network, images_batch[:1])

The convolutional layer has only 320 parameters: 32 filters × (3 × 3 weights + 1 bias). Almost all of the parameters are in the first `Linear` layer after flattening.

## Train the model

Same loss function, a new optimizer for the new model's parameters, and **exactly the same** `train` function.

In [ ]:
optimizer = torch.optim.Adam(cnn_network.parameters(), lr=0.001)

cnn_history = train(cnn_network, train_loader_cnn, loss_fn, optimizer, epochs=5)

In [ ]:
epochs = range(1, len(cnn_history['loss']) + 1)

plt.plot(epochs, cnn_history['loss'], 'bo', label='Training loss')
plt.plot(epochs, cnn_history['accuracy'], 'b', label='Training accuracy')
plt.title('Training loss vs accuracy')
plt.xlabel('Epochs')
plt.ylabel('Loss/Accuracy')
plt.xticks(epochs)
plt.legend()
plt.show()

## Look at the learned filters

The layers of an `nn.Sequential` can be accessed by index. Layer 0 is the convolutional layer. Its weights have shape `(out_channels, in_channels, height, width)`: 32 filters, each 1×3×3.

In [ ]:
cnn_network[0]

In [ ]:
cnn_network[0].weight.shape

In [ ]:
cnn_network[0].bias.shape

In [ ]:
filters = cnn_network[0].weight.detach().cpu()

fig, ax = plt.subplots(4, 8, figsize=(8, 5))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.imshow(filters[i, 0], cmap='binary')
    plt.axis('off')
fig.tight_layout()
plt.show()

`.detach()` gives a copy of the weights without gradient tracking, and `.cpu()` moves them to the CPU, which matplotlib needs.

### What does each filter respond to?

Here are the 32 feature maps the convolutional layer produces for one test digit (after the ReLU). Each one highlights different strokes and edges.

In [ ]:
digit = test_images_cnn[4]

cnn_network.eval()
with torch.no_grad():
    feature_maps = cnn_network[1](cnn_network[0](digit.unsqueeze(0).to(device))).cpu()

feature_maps.shape

In [ ]:
fig, ax = plt.subplots(4, 8, figsize=(10, 5.5))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.imshow(feature_maps[0, i], cmap='viridis')
    plt.axis('off')
fig.tight_layout()
plt.show()

## Evaluate the model using the test data

In [ ]:
test_loss, test_acc = evaluate(cnn_network, test_loader_cnn, loss_fn)
print('test_loss:', test_loss)
print('test_acc:', test_acc)

A single test image now needs shape `(1, 1, 28, 28)`: a batch of one image with one channel.

In [ ]:
digit = test_images_cnn[4]
digit_label = test_labels[4]

plt.imshow(digit.reshape(28, 28), cmap=plt.cm.binary)
plt.show()
print(digit_label)

In [ ]:
digit.shape, digit.unsqueeze(0).shape

In [ ]:
with torch.no_grad():
    logits = cnn_network(digit.unsqueeze(0).to(device))

probabilities = torch.softmax(logits, dim=1)
print(probabilities.round(decimals=4))
print('predicted digit:', probabilities.argmax(dim=1).item())

In [ ]:
with torch.no_grad():
    logits = cnn_network(test_images_cnn[:5].to(device))

for p in torch.softmax(logits, dim=1):
    print(p.argmax().item())

In [ ]:
pred_labels_cnn = predict(cnn_network, test_loader_cnn)
y_pred_cnn = pred_labels_cnn.numpy()

cm_cnn = sklearn.metrics.confusion_matrix(y_true, y_pred_cnn)
cm_cnn

In [ ]:
print(f"Accuracy: {sklearn.metrics.accuracy_score(y_true, y_pred_cnn):.2%}")
print(f"Precision: {sklearn.metrics.precision_score(y_true, y_pred_cnn, average='macro'):.2%}")
print(f"Recall: {sklearn.metrics.recall_score(y_true, y_pred_cnn, average='macro'):.2%}")

In [ ]:
print(sklearn.metrics.classification_report(y_true, y_pred_cnn))

In [ ]:
sklearn.metrics.ConfusionMatrixDisplay(cm_cnn).plot(cmap='Blues', colorbar=False)
plt.title('CNN')
plt.show()

show_mistakes(y_true, y_pred_cnn, 'CNN')

# Part 3: Dense network vs CNN

In [ ]:
dense_params = sum(p.numel() for p in dense_network.parameters())
cnn_params = sum(p.numel() for p in cnn_network.parameters())

print(f"{'':10s} {'parameters':>12s} {'test accuracy':>15s} {'mistakes':>10s}")
print(f"{'Dense':10s} {dense_params:12,d} {sklearn.metrics.accuracy_score(y_true, y_pred):15.2%} {(y_pred != y_true).sum():10d}")
print(f"{'CNN':10s} {cnn_params:12,d} {sklearn.metrics.accuracy_score(y_true, y_pred_cnn):15.2%} {(y_pred_cnn != y_true).sum():10d}")

The CNN makes fewer mistakes than the dense network, with a similar number of parameters. The gap is modest because this CNN has only one convolutional layer: almost all of its parameters are still in the dense layer after `Flatten`. Stacking more convolutional layers widens the gap.

### What happens if the digits move?

All the MNIST digits are centered in their images. Let's shift every test image a few pixels to the right and see how each model copes. `torch.roll` shifts the pixels; the borders of MNIST images are blank, so nothing important gets cut off.

The dense network learned one weight per pixel *position*, so shifting the digit changes which weights it uses. The CNN's filters work the same way at every position.

In [ ]:
shifts = [0, 1, 2, 3, 4]
dense_acc, cnn_acc = [], []

for s in shifts:
    shifted = torch.roll(test_images, shifts=s, dims=2)      # dims=2 is the width
    shifted_flat = shifted.reshape(10000, 28 * 28).float() / 255
    shifted_cnn = shifted.reshape(10000, 1, 28, 28).float() / 255

    p_dense = predict(dense_network, DataLoader(TensorDataset(shifted_flat, test_labels), batch_size=1000))
    p_cnn = predict(cnn_network, DataLoader(TensorDataset(shifted_cnn, test_labels), batch_size=1000))
    dense_acc.append((p_dense == test_labels).float().mean().item())
    cnn_acc.append((p_cnn == test_labels).float().mean().item())
    print(f"shift {s}:   dense {dense_acc[-1]:.3f}   CNN {cnn_acc[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, s in zip(axes, shifts):
    ax.imshow(torch.roll(test_images[0], shifts=s, dims=1), cmap=plt.cm.binary)
    ax.set_title(f"shift {s}")
    ax.axis('off')
plt.show()

plt.plot(shifts, dense_acc, 'o-', label='Dense')
plt.plot(shifts, cnn_acc, 'o-', label='CNN')
plt.xticks(shifts)
plt.xlabel('Shift to the right (pixels)')
plt.ylabel('Test accuracy')
plt.title('Accuracy on shifted test digits')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Neither model was trained on shifted digits, so both get worse, but the CNN holds up better.

# Summary

* **Classification in PyTorch:** the last layer has one output per class; the model outputs **logits** (no softmax layer); the labels are plain integers; the loss is `nn.CrossEntropyLoss`. Use `argmax` to get predicted classes, and `torch.softmax` when you want probabilities.
* **Image data** has shape `(batch, channels, height, width)`. A dense network needs each image flattened to a vector; a CNN takes the 2-D images directly.
* A **convolutional layer** slides small learned filters across the image. It reuses the same weights at every position, so it needs few parameters and detects a pattern wherever it appears. A **max pooling** layer shrinks the feature maps.
* The **training loop was the same** for both models, and the same as for housing prices in the previous notebook. Only the model and the shape of the data changed.